# corrector_05 - Clean the corpus (role B, task item 6)

1. Measure the overlap of the **current** corpus (train.txt against val.txt and test.txt) with the
   normalization of `scripts/check_text_overlap.py` (lower case, punctuation removed), and save the
   list of train.txt line numbers that duplicate a val or test sentence.
2. Keep the current corpus as `data/corpus_v1/` (nothing is deleted).
3. Rebuild the corpus into `data/corpus/` with the fixed rules of `src/corrector/corpus.py`:
   4 to 40 syllables, no cut after an initial ("Ulysses S. Grant"), duplicates removed after
   normalization, and the split chosen from the normalized sentence, so near-duplicates always land
   in the same split.
4. Check again: the exact overlap must be 0. Also compare with the UIT-HWDB val and test lines.

The new `data/corpus/` is the main corpus from now on. Every model trained on the old one is trained
again in `corrector_06_final.ipynb`. CPU runtime is enough (about 15 minutes).

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GITHUB_TOKEN").strip()
BRANCH = "b/corrector"  # change to "main" after the code is merged
!rm -rf /content/repo
!git clone -q -b {BRANCH} https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

Mounted at /content/drive
/content/repo


In [ ]:
import json, os, shutil, subprocess, sys
from src.corrector.corpus import corpus_overlap

CORPUS_DIR = f"{DATA_DIR}/corpus"
V1_DIR = f"{DATA_DIR}/corpus_v1"
MAX_SENTENCES = 1_500_000
os.makedirs(OUT_DIR, exist_ok=True)
report = {}

## 1. Overlap of the current corpus, and the lines to remove

In [ ]:
source_dir = V1_DIR if os.path.exists(f"{V1_DIR}/train.txt") else CORPUS_DIR
old = corpus_overlap(f"{source_dir}/train.txt",
                     {"val": f"{source_dir}/val.txt", "test": f"{source_dir}/test.txt"}, window=8)
report["v1_overlap"] = {k: v for k, v in old.items() if k != "train_lines_to_remove"}
report["v1_train_lines_removed"] = old["train_lines_to_remove"]
print("current corpus (v1):", json.dumps(report["v1_overlap"], indent=1))
print("train.txt lines that repeat a val/test sentence:", len(old["train_lines_to_remove"]))
with open(f"{OUT_DIR}/corrector_corpus_removed_lines.json", "w", encoding="utf-8") as f:
    json.dump({"file": "corpus_v1/train.txt", "line_numbers_1_based": old["train_lines_to_remove"],
               "rule": "normalized text (scripts/check_text_overlap.py) equal to a val or test sentence"},
              f, indent=1)

current corpus (v1): {
 "window": 8,
 "val": {
  "lines": 15057,
  "exact": 119,
  "ngram_any": 2340
 },
 "test": {
  "lines": 15061,
  "exact": 125,
  "ngram_any": 2367
 }
}
train.txt lines that repeat a val/test sentence: 334


## 2. Keep the current corpus as corpus_v1

In [ ]:
if not os.path.exists(f"{V1_DIR}/train.txt"):
    shutil.copytree(CORPUS_DIR, V1_DIR, dirs_exist_ok=True)
    print("copied", CORPUS_DIR, "->", V1_DIR)
else:
    print("corpus_v1 already exists, kept as it is")

copied /content/drive/MyDrive/Vietnamese-Handwriting-OCR/data/corpus -> /content/drive/MyDrive/Vietnamese-Handwriting-OCR/data/corpus_v1


## 3. Rebuild the corpus (Vietnamese Wikipedia, 20231101.vi)

In [ ]:
from datasets import load_dataset
from src.corrector.corpus import build_corpus

LOCAL_DIR = "/content/corpus_v2"  # write locally first (fast), copy to Drive at the end
wiki = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train", streaming=True)
stats = build_corpus((article["text"] for article in wiki), LOCAL_DIR, MAX_SENTENCES)

syllables = 0
with open(f"{LOCAL_DIR}/train.txt", encoding="utf-8") as f:
    for line in f:
        syllables += len(line.split())
stats.update(source="wikimedia/wikipedia 20231101.vi", train_tokens=syllables,
             license="CC BY-SA 3.0 and GFDL (dataset card)")
with open(f"{LOCAL_DIR}/stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)
for name in ["train.txt", "val.txt", "test.txt", "stats.json"]:
    shutil.copy(f"{LOCAL_DIR}/{name}", f"{CORPUS_DIR}/{name}")
report["v2_stats"] = stats
print(json.dumps(stats, ensure_ascii=False, indent=2))

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

   20000 articles     947448 sentences     335 s
   40000 articles    1365941 sentences     487 s
{
  "articles_read": 54860,
  "sentences": {
    "train": 1469903,
    "val": 15122,
    "test": 15028
  },
  "duplicates_skipped": 159875,
  "min_syllables": 4,
  "max_syllables": 40,
  "built_at": "2026-11-22 08:53:31",
  "source": "wikimedia/wikipedia 20231101.vi",
  "train_tokens": 29617815,
  "license": "CC BY-SA 3.0 and GFDL (dataset card)"
}


## 4. Check again (expected: exact overlap 0) and compare with UIT-HWDB

In [ ]:
new = corpus_overlap(f"{CORPUS_DIR}/train.txt",
                     {"val": f"{CORPUS_DIR}/val.txt", "test": f"{CORPUS_DIR}/test.txt"}, window=8)
report["v2_overlap"] = {k: v for k, v in new.items() if k != "train_lines_to_remove"}
print("new corpus:", json.dumps(report["v2_overlap"], indent=1))
assert new["val"]["exact"] == 0 and new["test"]["exact"] == 0, "exact overlap is not 0"

result = subprocess.run([sys.executable, "scripts/check_text_overlap.py",
                         "--index", f"{OUT_DIR}/dataset_index.jsonl", "--splits", f"{OUT_DIR}/splits.json",
                         "--corpus", f"{CORPUS_DIR}/train.txt"], capture_output=True, text=True)
print(result.stdout[-3000:])
report["uit_hwdb_vs_corpus_stdout"] = result.stdout[-3000:]

with open(f"{OUT_DIR}/corrector_corpus_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1)
print("saved", f"{OUT_DIR}/corrector_corpus_report.json")

new corpus: {
 "window": 8,
 "val": {
  "lines": 15122,
  "exact": 0,
  "ngram_any": 2201
 },
 "test": {
  "lines": 15028,
  "exact": 0,
  "ngram_any": 2176
 }
}
text overlap report (window = 5 words)
lines per split: train=5369, val=636, test=201

against the training text (5369 lines)
  split   lines     exact       any     major  eligible
  val       636      0.2%      4.7%      0.2%       621
  test      201      0.0%      5.7%      2.1%       192
verdict val: KEEP (major = 0.2%): keep the split and note the overlap in the report
verdict test: KEEP (major = 2.1%): keep the split and note the overlap in the report

  most overlapping val pages:
    group     lines    exact      any    major  top train page
    85           28     0.0%    10.7%     3.6%  226 (8 windows)
    51           46     0.0%     2.2%     0.0%  47 (3 windows) +2 tied
    42           42     0.0%    17.1%     0.0%  223 (8 windows)
    176          40     0.0%     0.0%     0.0%  -
    77           36     0.0%    